# SmartBite SVTRv2 Expiry-Date Recognition Fine-Tuning (Official PaddleOCR)

This notebook trains SVTRv2 using the official PaddleOCR SVTRv2 config path, not PaddleX `Engine`.

Why: PaddleX exposes `ch_SVTRv2_rec` for inference, but in current Colab/PaddleX installs it is not registered as a trainable PaddleX model. The official SVTRv2 training docs use `configs/rec/SVTRv2/rec_svtrv2_ch.yml`, so this notebook follows that route directly.

Defaults are set for L4: batch size 128, 50 epochs, augmentation off for the first clean baseline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('PYTHONUNBUFFERED', '1')


def run_live(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'command failed with exit code {rc}: {shlex.join(cmd)}')
    return rc


## Config


In [ ]:
DATASET_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/products_date_recognition_balanced.zip')
OUTPUT_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/svtrv2_expdate_finetune_paddleocr')

WORK = Path('/content/smartbite_svtrv2_paddleocr')
PADDLEOCR_DIR = WORK / 'PaddleOCR'
BASE_UNZIP_DIR = WORK / 'dataset_unzip'
DATA_DIR = WORK / 'svtrv2_data'
OUTPUT_DIR = WORK / 'output'
TRAIN_OUTPUT_DIR = OUTPUT_DIR / 'train'
EXPORT_DIR = OUTPUT_DIR / 'inference' / 'smartbite_svtrv2_expdate_rec'
CONFIG_PATH = WORK / 'smartbite_svtrv2_expdate.yml'
DICT_PATH = WORK / 'smartbite_expdate_dict.txt'
PRETRAINED_PATH = WORK / 'ch_SVTRv2_rec_pretrained.pdparams'

EPOCHS = 50
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 1e-4
DEVICE = 'gpu:0'

ENABLE_MG_STYLE_AUGMENTATION = False
AUG_PER_IMAGE = 2
AUG_MAX_TRAIN_SAMPLES = None

SMARTBITE_CHARS = list('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ/.-:ÜİŞĞÇÖ')

print('dataset zip:', DATASET_ZIP_DRIVE)
print('output drive dir:', OUTPUT_DRIVE_DIR)
print('batch size:', BATCH_SIZE, 'epochs:', EPOCHS, 'device:', DEVICE)


## Install Paddle + Clone PaddleOCR


In [ ]:
WORK.mkdir(parents=True, exist_ok=True)

# Avoid Colab's broken torch/NCCL stack affecting unrelated imports. PaddleOCR training does not need torch.
run_live([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'])

try:
    run_live([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle-gpu==3.0.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'])
except Exception as exc:
    print('GPU Paddle install failed, falling back to CPU Paddle:', exc)
    run_live([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.0.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/'])

run_live([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'opencv-python-headless', 'requests'])

if not PADDLEOCR_DIR.exists():
    run_live(['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', PADDLEOCR_DIR])
else:
    print('PaddleOCR already exists:', PADDLEOCR_DIR)

run_live([sys.executable, '-m', 'pip', 'install', '-q', '-r', PADDLEOCR_DIR / 'requirements.txt'])

# PaddleOCR has used both old and new SVTRv2 config names across docs/releases.
# Current docs/link-checker references ch_SVTRv2_rec.yml; older docs referenced rec_svtrv2_ch.yml.
SVTRV2_CONFIG_CANDIDATES = [
    'configs/rec/SVTRv2/ch_SVTRv2_rec.yml',
    'configs/rec/SVTRv2/rec_svtrv2_ch.yml',
    'configs/rec/SVTRv2/ch_SVTRv2_rec_distillation.yml',
    'configs/rec/SVTRv2/rec_svtrv2_ch_distillation.yml',
]
SVTRV2_CONFIG_PATH = None
for rel in SVTRV2_CONFIG_CANDIDATES:
    path = PADDLEOCR_DIR / rel
    print(rel, path.exists())
    if path.exists() and SVTRV2_CONFIG_PATH is None:
        SVTRV2_CONFIG_PATH = path

if SVTRV2_CONFIG_PATH is None:
    print('SVTR/SVTRv2 configs found in this clone:')
    run_live(['find', PADDLEOCR_DIR / 'configs' / 'rec', '-maxdepth', '4', '-iname', '*svtr*'])
    raise AssertionError('No SVTRv2 config found. The cloned PaddleOCR branch does not include SVTRv2 training configs.')

print('Using SVTRv2 config:', SVTRV2_CONFIG_PATH)

## Unzip SmartBite Recognition Dataset


In [ ]:
assert DATASET_ZIP_DRIVE.exists(), f'Missing dataset zip: {DATASET_ZIP_DRIVE}'
if BASE_UNZIP_DIR.exists():
    shutil.rmtree(BASE_UNZIP_DIR)
BASE_UNZIP_DIR.mkdir(parents=True, exist_ok=True)
run_live(['unzip', '-q', '-o', DATASET_ZIP_DRIVE, '-d', BASE_UNZIP_DIR])


def find_dataset_root(base: Path) -> Path:
    candidates = [p.parent for p in base.rglob('train_label.txt') if (p.parent / 'val_label.txt').exists()]
    assert candidates, f'No dataset root with train_label.txt and val_label.txt under {base}'
    candidates.sort(key=lambda p: len(str(p)))
    return candidates[0]

DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
print('DATASET_ROOT:', DATASET_ROOT)
for name in ['train_label.txt', 'val_label.txt', 'test_label.txt']:
    path = DATASET_ROOT / name
    if path.exists():
        print(name, sum(1 for line in path.read_text(encoding='utf-8').splitlines() if line.strip()))


## Convert Dataset To PaddleOCR Recognition Layout


In [ ]:
import re
import unicodedata
from collections import Counter


def normalize_label(text: str) -> str:
    text = unicodedata.normalize('NFKC', text).strip().upper()
    text = re.sub(r'\s+', ' ', text)
    return text


def read_label_file(path: Path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip() or '\t' not in line:
            continue
        rel, text = line.split('\t', 1)
        rows.append((rel.strip(), normalize_label(text)))
    return rows


def copy_rows(rows, split_name):
    out_rows = []
    img_dir = DATA_DIR / f'{split_name}_images'
    img_dir.mkdir(parents=True, exist_ok=True)
    for idx, (rel, text) in enumerate(rows):
        src = DATASET_ROOT / rel
        if not src.exists():
            src = DATASET_ROOT / Path(rel).name
        if not src.exists():
            continue
        suffix = src.suffix.lower() or '.jpg'
        dst_rel = f'{split_name}_images/{split_name}_{idx:06d}{suffix}'
        dst = DATA_DIR / dst_rel
        shutil.copy2(src, dst)
        out_rows.append((dst_rel, text))
    return out_rows


train_rows_raw = read_label_file(DATASET_ROOT / 'train_label.txt')
val_rows_raw = read_label_file(DATASET_ROOT / 'val_label.txt')
test_rows_raw = read_label_file(DATASET_ROOT / 'test_label.txt') if (DATASET_ROOT / 'test_label.txt').exists() else []
print('raw rows:', {'train': len(train_rows_raw), 'val': len(val_rows_raw), 'test': len(test_rows_raw)})
print('sample labels:', train_rows_raw[:5])

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_rows = copy_rows(train_rows_raw, 'train')
val_rows = copy_rows(val_rows_raw, 'val')
test_rows = copy_rows(test_rows_raw, 'test') if test_rows_raw else []

DICT_PATH.write_text('\n'.join(dict.fromkeys(SMARTBITE_CHARS)) + '\n', encoding='utf-8')
print('dict chars:', len(SMARTBITE_CHARS), ''.join(SMARTBITE_CHARS))


def write_rows(path: Path, rows):
    path.write_text(''.join(f'{rel}\t{text}\n' for rel, text in rows), encoding='utf-8')


write_rows(DATA_DIR / 'train_label.txt', train_rows)
write_rows(DATA_DIR / 'val_label.txt', val_rows)
if test_rows:
    write_rows(DATA_DIR / 'test_label.txt', test_rows)

all_text = ''.join(text for _, text in train_rows + val_rows + test_rows)
unknown = sorted(set(all_text) - set(SMARTBITE_CHARS) - {' '})
print('converted rows:', {'train': len(train_rows), 'val': len(val_rows), 'test': len(test_rows)})
print('unknown chars not in dict:', unknown[:50])
assert not unknown, f'Labels contain chars missing from SMARTBITE_CHARS: {unknown}' 

## Optional SVTR-MG-Style Augmentation


In [ ]:
import cv2
import numpy as np
import random

rng = random.Random(42)


def aug_low_contrast(img):
    alpha = rng.uniform(0.45, 0.75)
    beta = rng.randint(8, 35)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def aug_blur_downsample(img):
    h, w = img.shape[:2]
    scale = rng.uniform(0.45, 0.75)
    small = cv2.resize(img, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
    up = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if rng.random() < 0.6:
        up = cv2.GaussianBlur(up, (3, 3), 0)
    return up


def aug_broken_strokes(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if rng.random() < 0.5:
        gray = cv2.erode(gray, np.ones((2, 2), np.uint8), iterations=1)
    noise = (np.random.default_rng(rng.randint(0, 10**9)).random(gray.shape) > 0.985).astype(np.uint8) * 255
    gray = cv2.max(gray, noise)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

AUGS = [aug_low_contrast, aug_blur_downsample, aug_broken_strokes]
aug_rows = []
if ENABLE_MG_STYLE_AUGMENTATION:
    aug_dir = DATA_DIR / 'train_images_mg_aug'
    aug_dir.mkdir(parents=True, exist_ok=True)
    source_rows = train_rows[:AUG_MAX_TRAIN_SAMPLES] if AUG_MAX_TRAIN_SAMPLES else train_rows
    for idx, (rel, text) in enumerate(source_rows):
        img = cv2.imread(str(DATA_DIR / rel), cv2.IMREAD_COLOR)
        if img is None:
            continue
        for aug_i in range(AUG_PER_IMAGE):
            aug = rng.choice(AUGS)(img)
            dst_rel = f'train_images_mg_aug/aug_{idx:06d}_{aug_i}.jpg'
            cv2.imwrite(str(DATA_DIR / dst_rel), aug)
            aug_rows.append((dst_rel, text))
    train_rows = train_rows + aug_rows
    write_rows(DATA_DIR / 'train_label.txt', train_rows)
print('augmentation enabled:', ENABLE_MG_STYLE_AUGMENTATION, 'aug rows:', len(aug_rows), 'train total:', len(train_rows))


## Download SVTRv2 Training Weights


In [ ]:
import requests

# PaddleOCR v3 text-recognition docs expose this as the ch_SVTRv2_rec pretrained/training model.
PRETRAINED_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/ch_SVTRv2_rec_pretrained.pdparams'
if not PRETRAINED_PATH.exists():
    print('downloading:', PRETRAINED_URL)
    with requests.get(PRETRAINED_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with PRETRAINED_PATH.open('wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
print('PRETRAINED_PATH:', PRETRAINED_PATH, PRETRAINED_PATH.stat().st_size)


## Create Official PaddleOCR SVTRv2 Config


In [ ]:
import yaml

BASE_CONFIG = SVTRV2_CONFIG_PATH
base_cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
print('BASE_CONFIG:', BASE_CONFIG)

cfg = base_cfg
cfg['Global']['use_gpu'] = DEVICE.startswith('gpu')
cfg['Global']['epoch_num'] = int(EPOCHS)
cfg['Global']['save_model_dir'] = str(TRAIN_OUTPUT_DIR)
cfg['Global']['save_epoch_step'] = 1
cfg['Global']['eval_batch_step'] = [0, 200]
cfg['Global']['print_batch_step'] = 10
cfg['Global']['pretrained_model'] = str(PRETRAINED_PATH.with_suffix('')) if PRETRAINED_PATH.suffix == '.pdparams' else str(PRETRAINED_PATH)
cfg['Global']['checkpoints'] = None
cfg['Global']['save_inference_dir'] = str(EXPORT_DIR)
cfg['Global']['infer_img'] = str(DATA_DIR / val_rows[0][0]) if val_rows else ''
cfg['Global']['character_dict_path'] = str(DICT_PATH)
cfg['Global']['max_text_length'] = 25
cfg['Global']['use_space_char'] = True
cfg['Global']['distributed'] = False
cfg['Global']['save_res_path'] = str(OUTPUT_DIR / 'predicts_svtrv2.txt')

cfg['Optimizer']['lr']['learning_rate'] = float(LEARNING_RATE)
if 'warmup_epoch' in cfg['Optimizer']['lr']:
    cfg['Optimizer']['lr']['warmup_epoch'] = 1

cfg['Train']['dataset']['data_dir'] = str(DATA_DIR)
cfg['Train']['dataset']['label_file_list'] = [str(DATA_DIR / 'train_label.txt')]
cfg['Train']['loader']['batch_size_per_card'] = int(BATCH_SIZE)
cfg['Train']['loader']['num_workers'] = 2
cfg['Train']['loader']['drop_last'] = True
if 'sampler' in cfg['Train']:
    cfg['Train']['sampler']['first_bs'] = int(BATCH_SIZE)

cfg['Eval']['dataset']['data_dir'] = str(DATA_DIR)
cfg['Eval']['dataset']['label_file_list'] = [str(DATA_DIR / 'val_label.txt')]
cfg['Eval']['loader']['batch_size_per_card'] = int(EVAL_BATCH_SIZE)
cfg['Eval']['loader']['num_workers'] = 2
cfg['Eval']['loader']['drop_last'] = False

# Sanity check: do not accidentally train the old PP-HGNet recognizer.
arch = cfg.get('Architecture', {})
backbone = arch.get('Backbone', {}) if isinstance(arch, dict) else {}
print('Architecture.algorithm:', arch.get('algorithm'))
print('Backbone.name:', backbone.get('name'))
assert backbone.get('name') == 'SVTRv2', f'Expected Backbone.name=SVTRv2, got {backbone.get("name")}'

CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding='utf-8')[:5000])

## Train SVTRv2


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_cmd = [sys.executable, '-u', PADDLEOCR_DIR / 'tools' / 'train.py', '-c', CONFIG_PATH]
run_live(train_cmd, cwd=PADDLEOCR_DIR)


## Locate Best Checkpoint


In [ ]:
all_pdparams = sorted(TRAIN_OUTPUT_DIR.rglob('*.pdparams'), key=lambda p: p.stat().st_mtime, reverse=True)
print('recent pdparams:')
for item in all_pdparams[:20]:
    print(item)

best_candidates = [p for p in all_pdparams if p.name == 'best_accuracy.pdparams']
if not best_candidates:
    best_candidates = [p for p in all_pdparams if 'best' in str(p).lower()]
if not best_candidates:
    best_candidates = all_pdparams
assert best_candidates, f'No checkpoints found under {TRAIN_OUTPUT_DIR}'
BEST_PD = best_candidates[0]
BEST_PREFIX = str(BEST_PD.with_suffix(''))
print('BEST_PD:', BEST_PD)
print('BEST_PREFIX:', BEST_PREFIX)


## Evaluate Best Checkpoint


In [ ]:
eval_cmd = [
    sys.executable, '-u', PADDLEOCR_DIR / 'tools' / 'eval.py',
    '-c', CONFIG_PATH,
    '-o', f'Global.pretrained_model={BEST_PREFIX}',
]
run_live(eval_cmd, cwd=PADDLEOCR_DIR)


## Export Inference Model


In [ ]:
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
export_cmd = [
    sys.executable, '-u', PADDLEOCR_DIR / 'tools' / 'export_model.py',
    '-c', CONFIG_PATH,
    '-o', f'Global.pretrained_model={BEST_PREFIX}', f'Global.save_inference_dir={EXPORT_DIR}',
]
run_live(export_cmd, cwd=PADDLEOCR_DIR)

print('exported files:')
for item in sorted(EXPORT_DIR.rglob('*')):
    if item.is_file():
        print(item.relative_to(EXPORT_DIR), item.stat().st_size)


## Package Artifacts To Drive


In [ ]:
OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
model_drive_dir = OUTPUT_DRIVE_DIR / 'smartbite_svtrv2_expdate_rec'
if model_drive_dir.exists():
    shutil.rmtree(model_drive_dir)
shutil.copytree(EXPORT_DIR, model_drive_dir)

for extra in [CONFIG_PATH, DICT_PATH]:
    if extra.exists():
        shutil.copy2(extra, OUTPUT_DRIVE_DIR / extra.name)

zip_base = OUTPUT_DRIVE_DIR / 'smartbite_svtrv2_expdate_rec'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=model_drive_dir)
print('Drive model dir:', model_drive_dir)
print('Drive zip:', zip_path)
print('Use locally as SMARTBITE_SVTRV2_REC_MODEL_DIR=models/svtrv2/smartbite_svtrv2_expdate_rec after unzipping.')


## Local Integration Notes

After downloading the zip locally:

```bash
mkdir -p models/svtrv2/smartbite_svtrv2_expdate_rec
unzip -o smartbite_svtrv2_expdate_rec.zip -d models/svtrv2/smartbite_svtrv2_expdate_rec
SMARTBITE_EXPIRY_RECOGNIZER=svtrv2 \
SMARTBITE_SVTRV2_REC_MODEL_NAME=ch_SVTRv2_rec \
SMARTBITE_SVTRV2_REC_MODEL_DIR=models/svtrv2/smartbite_svtrv2_expdate_rec \
uv run python app/scripts/manual_crop_recognition_benchmark.py --recognizer svtrv2
```
